# Train a small 1-D CNN over the 3LC table

Plain PyTorch, no Lightning, no Trainer. The 3LC table from notebook 2
is wrapped in a `torch.utils.data.Dataset` and consumed by a tiny
1-D convnet that maps a 3-second waveform to a binary class
(`chainsaw` vs `environment`).

Kept deliberately minimal — metric collection back into the table is
the next thing to wire in.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import tlc

In [2]:
N_ROWS = 10_000

train_table = tlc.Table.from_names(
    table_name=f"train-{N_ROWS}",
    dataset_name="frugalai",
    project_name="audio-classification-poc",
)
val_table = tlc.Table.from_names(
    table_name=f"val-{N_ROWS}",
    dataset_name="frugalai",
    project_name="audio-classification-poc",
)
print(f"train: {len(train_table)}  val: {len(val_table)}")

label_map = train_table.row_schema.values['label'].value.map
label_names = [label_map[float(i)]['internal_name'] for i in sorted(int(k) for k in label_map)]
print(f"labels: {label_names}")

train: 10000  val: 10000
labels: ['chainsaw', 'environment']


### `torch.utils.data.Dataset` over the 3LC table

Each `table[i]` triggers a WAV decode (handled by the `wav_audio` sample type)
and returns a NumPy float32 array. We crop/pad to a fixed length so the
DataLoader can batch them, and convert to a tensor.

The dataset returns a **3-tuple** `(x, y, example_id)` where `example_id` is
the row index in the underlying 3LC table. The training loop ignores it; the
eval loop uses it as the foreign key when writing per-sample metrics back to
the table via `MetricsTableWriter`.

In [3]:
FIXED_LEN = 12000 * 3  # 3 seconds @ 12 kHz


class TlcAudioDataset(Dataset):
    """Adapter over a 3LC table.

    Returns ``(x, y, example_id)`` where ``example_id`` is the row index in
    the underlying table. We carry the index through the DataLoader so the
    eval pass can write per-sample metrics keyed back to the foreign table.
    """

    def __init__(self, table, fixed_len=FIXED_LEN):
        self.table = table
        self.fixed_len = fixed_len

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx):
        sample = self.table[idx]
        wav = sample['audio'].astype(np.float32)
        if wav.shape[0] >= self.fixed_len:
            wav = wav[: self.fixed_len]
        else:
            wav = np.pad(wav, (0, self.fixed_len - wav.shape[0]))
        return torch.from_numpy(wav).unsqueeze(0), int(sample['label']), idx


train_ds = TlcAudioDataset(train_table)
val_ds = TlcAudioDataset(val_table)
x, y, ex_id = train_ds[0]
print(f"x: {tuple(x.shape)}, dtype={x.dtype}; y: {y}; example_id: {ex_id}")
print(f"train: {len(train_ds)}  val: {len(val_ds)}")

x: (1, 36000), dtype=torch.float32; y: 1; example_id: 0
train: 10000  val: 10000


### Tiny 1-D CNN

Strided conv stack over the raw waveform → global pool → linear. Few
parameters, trains in seconds on CPU. The point here is to prove the
data path end-to-end, not chase accuracy.

In [4]:
class TinyAudioCNN(nn.Module):
    """Backbone + head, with an explicit embedding accessor.

    ``forward`` returns ``(logits, embedding)`` so the eval loop can write
    embeddings to the metrics table without re-running the backbone or
    fiddling with hooks.
    """

    def __init__(self, n_classes=2, embedding_dim=64):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=80, stride=16),  # 36000 -> ~2245
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),  # -> ~561
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),  # -> ~140
            nn.Conv1d(32, embedding_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(embedding_dim), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Linear(embedding_dim, n_classes)

    def forward(self, x):
        embedding = self.backbone(x).squeeze(-1)
        logits = self.head(embedding)
        return logits, embedding


model = TinyAudioCNN(n_classes=len(label_names))
print(model)
print("params:", sum(p.numel() for p in model.parameters()))

TinyAudioCNN(
  (backbone): Sequential(
    (0): Conv1d(1, 16, kernel_size=(80,), stride=(16,))
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (4): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool1d(kernel_size=4, stride=4, padding=0, dilation=1, ceil_mode=False)
    (8): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveAvgPool1d(output_size=1)
  )
  (head): Linear(in_features=64, out_features=2, bias=True)
)
params: 9426


### Training loop with per-sample metrics + embeddings

The eval pass uses a `MetricsTableWriter` to write per-sample metrics back
to the run, with `example_id` as the foreign key into the input table. Each
val sample contributes one row of metrics per epoch:

- `predicted` — argmax label (with the input table's class names attached)
- `confidence` — softmax probability of the predicted class
- `loss` — per-sample cross-entropy
- `embedding` — the 64-d feature from the layer just before the classifier head

The Dashboard can then sort/filter the input table by these metrics, and
the embeddings get UMAP-reduced to 2D after training (see the cell below).

This is the manual equivalent of `tlc.collect_metrics(table, collector, model)`
— more direct for cases where the table can't be passed as the dataset
(here, because we wrap it in a custom adapter).

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"device: {device}")
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

EPOCHS = 5

# Per-sample loss for metrics; reduce manually for the training step.
loss_fn_unreduced = nn.CrossEntropyLoss(reduction='none')

# `predicted` shares the input table's class names so the Dashboard renders
# labels rather than raw integers. Confidence and loss render as floats.
# `embedding` is the post-pool feature; we'll reduce it to 2D after training.
embedding_dim = model.head.in_features
metrics_schema = {
    'predicted': tlc.schemas.CategoricalLabelSchema(
        classes=label_names,
        display_name='predicted label',
        description='Argmax of the classifier output.',
        writable=False,
    ),
    'confidence': tlc.schemas.ConfidenceSchema(
        description='Softmax probability of the predicted class.',
    ),
    'loss': tlc.schemas.Float32Schema(
        display_name='loss',
        description='Per-sample cross-entropy loss.',
        writable=False,
    ),
    'embedding': tlc.schemas.EmbeddingSchema(
        display_name='embedding',
        description='Pre-head feature vector.',
        shape=(embedding_dim,),
    ),
}

run = tlc.init(
    project_name='audio-classification-poc',
    run_name='tiny-cnn',
    description=f'Tiny 1D CNN over rfcx/frugalai (train={len(train_ds)} val={len(val_ds)}).',
    if_exists='overwrite',
)

with tlc.MetricsTableWriter(
    run_url=run.url,
    foreign_table_url=val_table.url,
    foreign_table_display_name=val_table.dataset_name,
    schema=metrics_schema,
) as metrics_writer:
    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        for x, y, _ex in train_loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = loss_fn_unreduced(logits, y).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        correct = total = 0
        with torch.no_grad():
            for x, y, ex_ids in val_loader:
                x, y = x.to(device), y.to(device)
                logits, embeddings = model(x)
                losses = loss_fn_unreduced(logits, y)
                probs = F.softmax(logits, dim=-1)
                preds = probs.argmax(-1)
                conf = probs.gather(1, preds.unsqueeze(-1)).squeeze(-1)

                metrics_writer.add_batch({
                    tlc.constants.EXAMPLE_ID: ex_ids.tolist(),
                    tlc.constants.EPOCH: [epoch] * len(ex_ids),
                    'predicted': preds.cpu().numpy(),
                    'confidence': conf.cpu().numpy(),
                    'loss': losses.cpu().numpy(),
                    'embedding': embeddings.cpu().numpy(),
                })

                val_losses.append(losses.mean().item())
                correct += (preds == y).sum().item()
                total += y.numel()
 
        print(
            f"epoch {epoch + 1}/{EPOCHS}  "
            f"train_loss={np.mean(train_losses):.3f}  "
            f"val_loss={np.mean(val_losses):.3f}  "
            f"val_acc={correct / total:.3f}"
        )

print(f"\nrun: {run.url}")
run.set_status_completed()

device: mps


epoch 1/5  train_loss=0.567  val_loss=0.505  val_acc=0.744


epoch 2/5  train_loss=0.441  val_loss=0.456  val_acc=0.788


epoch 3/5  train_loss=0.407  val_loss=0.466  val_acc=0.778


epoch 4/5  train_loss=0.390  val_loss=0.699  val_acc=0.681


epoch 5/5  train_loss=0.369  val_loss=0.392  val_acc=0.814

run: /Users/gudbrand/Library/Application Support/3LC/projects/audio-classification-poc/runs/tiny-cnn


### Reduce embeddings to 2D with UMAP

`run.reduce_embeddings_by_foreign_table_url(...)` fits a single reducer
(default UMAP) on the most recently written metrics stream for the given
foreign table — i.e. the final epoch's val embeddings — and applies it to
every metrics table on the run. The reduced 2D coords are added as new
columns; source tables are deleted unless `delete_source_tables=False`.

In the Dashboard, those 2D coords give you a scatter view where you can
spot clusters, mislabels, and outliers visually.

In [6]:
url_mapping = run.reduce_embeddings_by_foreign_table_url(
    foreign_table_url=val_table.url,
    n_components=2,
)
print(f"reduced {len(url_mapping)} metrics table(s)")
for src, dst in url_mapping.items():
    print(f"  {src.name} -> {dst.name}")

/Users/gudbrand/Projects/3lc-examples/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Epochs completed:   0%|            0/200 [00:00]

reduced 1 metrics table(s)
  metrics_0000 -> reduced_0000


/Users/gudbrand/Projects/3lc-examples/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
